In [ ]:
from litellm import completion
import os
import openai
from typing import List, Dict, Optional
from dotenv import load_dotenv
import json
import sys
try:
    from pinecone import Pinecone, ServerlessSpec
except ImportError:
    print("Warning: Pinecone library not available. Install with: pip install pinecone")
    Pinecone = None
    ServerlessSpec = None
import hashlib

# Load environment variables from .env file
load_dotenv()

from getpass import getpass

# Get API keys from environment variables (safer approach)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("Error: OPENAI_API_KEY not found in environment variables")
    exit(1)

pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_environment = os.getenv("PINECONE_ENVIRONMENT", "us-east-1")  # Default to us-east-1
pinecone_cloud = os.getenv("PINECONE_CLOUD", "aws")  # Default to aws, can be aws, gcp, or azure

# Check if both API key and Pinecone library are available
if not pinecone_api_key:
    print("Warning: PINECONE_API_KEY not found. Vector DB features will be disabled.")
    use_pinecone = False
elif Pinecone is None or ServerlessSpec is None:
    print("Warning: Pinecone library not available. Vector DB features will be disabled.")
    use_pinecone = False
else:
    use_pinecone = True
    print("✅ Pinecone API key found - Vector DB features enabled")

def read_text_from_file(file_path: str) -> str:
    """
    Read text content from a file with size checking
   
    Args:
        file_path: Path to the text file
       
    Returns:
        Text content from the file (potentially truncated if too large)
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read().strip()
           
        # Check file size - OpenAI has token limits
        max_chars = 90000  # Much smaller limit to avoid 400 errors
       
        if len(content) > max_chars:
            print(f"⚠️  WARNING: File is large ({len(content):,} characters)")
            print(f"Truncating to first {max_chars:,} characters for processing...")
            print("Consider breaking large files into smaller chunks for better results.")
            content = content[:max_chars] + "\n\n[... FILE TRUNCATED FOR SIZE LIMITS ...]"
           
        return content
       
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        sys.exit(1)
    except Exception as e:
        print(f"Error reading file '{file_path}': {e}")
        sys.exit(1)

# Initialize Pinecone
pc = None
index = None
index_name = "trl-knowledge-base"

if use_pinecone:
    try:
        pc = Pinecone(api_key=pinecone_api_key)
       
        # Check if index exists, create if not
        existing_indexes = [idx.name for idx in pc.list_indexes()]
       
        if index_name not in existing_indexes:
            print(f"📦 Creating Pinecone index: {index_name}")
            print(f"   Cloud: {pinecone_cloud}, Region: {pinecone_environment}")
            pc.create_index(
                name=index_name,
                dimension=1536,  # OpenAI embedding dimension for text-embedding-ada-002
                metric="cosine",
                spec=ServerlessSpec(
                    cloud=pinecone_cloud,
                    region=pinecone_environment
                )
            )
            print(f"✅ Index '{index_name}' created successfully")
        else:
            print(f"✅ Connected to existing Pinecone index: {index_name}")
       
        index = pc.Index(index_name)
       
    except Exception as e:
        print(f"⚠️  Warning: Failed to initialize Pinecone: {e}")
        print("Continuing without vector DB support...")
        use_pinecone = False

def chunk_text(text: str, chunk_size: int = 1000, overlap: int = 200) -> List[str]:
    """
    Split text into overlapping chunks for embedding
   
    Args:
        text: Input text to chunk
        chunk_size: Size of each chunk in characters
        overlap: Overlap between chunks in characters
       
    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
   
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
   
    return chunks

def generate_embedding(text: str) -> List[float]:
    """
    Generate OpenAI embedding for text
   
    Args:
        text: Text to embed
       
    Returns:
        Embedding vector
    """
    try:
        response = openai.embeddings.create(
            input=text,
            model="text-embedding-ada-002"
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Error generating embedding: {e}")
        return None

def store_text_in_pinecone(text: str, metadata: Dict = None) -> bool:
    """
    Chunk text, generate embeddings, and store in Pinecone
   
    Args:
        text: Text to store
        metadata: Optional metadata to attach to vectors
       
    Returns:
        Success status
    """
    if not use_pinecone or not index:
        print("Pinecone not available")
        return False
   
    try:
        print("\n📝 Chunking text for vector storage...")
        chunks = chunk_text(text)
        print(f"Created {len(chunks)} chunks")
       
        print("🔢 Generating embeddings...")
        vectors = []
       
        for i, chunk in enumerate(chunks):
            # Generate unique ID for chunk
            chunk_id = hashlib.md5(chunk.encode()).hexdigest()
           
            # Generate embedding
            embedding = generate_embedding(chunk)
           
            if embedding:
                chunk_metadata = {
                    "text": chunk[:1000],  # Store first 1000 chars in metadata
                    "chunk_index": i,
                    "total_chunks": len(chunks)
                }
               
                if metadata:
                    chunk_metadata.update(metadata)
               
                vectors.append({
                    "id": chunk_id,
                    "values": embedding,
                    "metadata": chunk_metadata
                })
           
            # Progress indicator
            if (i + 1) % 5 == 0 or i == len(chunks) - 1:
                print(f"  Progress: {i + 1}/{len(chunks)} chunks processed")
       
        if vectors:
            print(f"\n💾 Storing {len(vectors)} vectors in Pinecone...")
            index.upsert(vectors=vectors)
            print("✅ Text successfully stored in vector database")
            return True
        else:
            print("❌ No vectors generated")
            return False
           
    except Exception as e:
        print(f"Error storing text in Pinecone: {e}")
        return False

def query_pinecone(query: str, top_k: int = 3) -> List[Dict]:
    """
    Query Pinecone for relevant context
   
    Args:
        query: Query text
        top_k: Number of results to return
       
    Returns:
        List of relevant text chunks with metadata
    """
    if not use_pinecone or not index:
        return []
   
    try:
        # Generate embedding for query
        query_embedding = generate_embedding(query)
       
        if not query_embedding:
            return []
       
        # Query Pinecone
        results = index.query(
            vector=query_embedding,
            top_k=top_k,
            include_metadata=True
        )
       
        # Extract relevant contexts
        contexts = []
        for match in results.matches:
            contexts.append({
                "text": match.metadata.get("text", ""),
                "score": match.score,
                "chunk_index": match.metadata.get("chunk_index", 0)
            })
       
        return contexts
       
    except Exception as e:
        print(f"Error querying Pinecone: {e}")
        return []

# Default file paths
default_trl_file = "12295881501_trl_sample.txt"
default_schema_file = "CAS-2.0-TAS-MMTEL-TRL-v.1.13.avsc"

# Check if user provided file paths as command line arguments
# Usage: python script.py <trl_file> <schema_file>
if len(sys.argv) > 2:
    trl_file = sys.argv[1]
    schema_file = sys.argv[2]
    print(f"TRL File with 2 Args: '{trl_file}'")
    print(f"Schema File: '{schema_file}'")
elif len(sys.argv) > 1:
    trl_file = sys.argv[1]
    schema_file = default_schema_file
    print(f"TRL File with 1 Arg: '{trl_file}'")
    print(f"Schema File: '{schema_file}' (default)")
else:
    trl_file = default_trl_file
    schema_file = default_schema_file
    print(f"Using default files:")
    print(f"  TRL File: '{trl_file}'")
    print(f"  Schema File: '{schema_file}'")

def get_user_question() -> str:
    """
    Get a question from the user via CLI prompt
   
    Returns:
        User's question as a string
    """
    print("\n" + "="*60)
    print("INTERACTIVE Q&A SYSTEM")
    print("="*60)
    print("Ask any question about the loaded text content.")
    print("Type 'quit' or 'exit' to stop.")
    print("="*60)
   
    while True:
        user_question = input("\nYour question: ").strip()
       
        if user_question.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            sys.exit(0)
       
        if user_question:
            return user_question
        else:
            print("Please enter a valid question.")

print(f"\nReading TRL data from: {trl_file}")
text = read_text_from_file(trl_file)

print(f"Reading schema from: {schema_file}")
schema = read_text_from_file(schema_file)

# Store text in Pinecone for RAG
if use_pinecone:
    print("\n🚀 Storing TRL and Schema data in Vector Database...")
    store_text_in_pinecone(
        text=text,
        metadata={"source": "trl_data", "file": trl_file}
    )
    store_text_in_pinecone(
        text=schema,
        metadata={"source": "schema", "file": schema_file}
    )

print("\n🔍 AUTOMATED SCHEMA ANALYSIS")
print("="*60)
print("Analyzing Encoded TRL file structure and identifying missing schema fields...")

# For automated schema analysis, use a pre-defined question
user_question = "Analyze this input encoded TRL json file and identify what fields are missing based on the provided schema."

chatgpt_url = "https://api.openai.com/v1/chat/completions"
chatgpt_headers = {
    "content-type": "application/json",
    "Authorization": f"Bearer {api_key}"
}

def estimate_tokens(text: str) -> int:
    """Rough estimation of token count (1 token ≈ 4 characters for English)"""
    return len(text) // 4

# Create a specialized prompt for schema field analysis
prompt = f"""You are a telecommunications data expert. Analyze the following TRL (Traffic Record Log) data and provide a comprehensive field analysis.

TEXT CONTENT:
{text}

USER'S QUESTION: {user_question}

TASK: Please analyze this TRL data and provide:

1. **All Field Names Found**: List every field name present in the data structure
2. **Missing Standard Fields**: Based on standard TRL/CDR schemas, identify fields that should be present but are missing
3. **Field Categories**: Categorize fields into:
   - Top-level fields
   - CDR Information fields  
   - Basic Information fields
   - SCCAS Data fields
   - Extension fields

4. **Schema Compliance**: Compare against standard telecommunications CDR/TRL schema expectations

Standard TRL fields that should typically be present included in below schema content file:
SCHEMA CONTENT:
{schema}

Please provide a detailed analysis of what fields are present vs. missing from the expected schema."""


# Check token count
estimated_tokens = estimate_tokens(prompt)
print(f"📊 Estimated tokens: {estimated_tokens:,}")

if estimated_tokens > 3500:  # Leave room for response
    print("⚠️  Token count is high - truncating further...")
    # Further reduce content if needed
    max_content_chars = 10100
    if len(text) > max_content_chars:
        text = text[:max_content_chars] + "\n[...FURTHER TRUNCATED...]"
        print(f"\nTruncated Text: {text}")
        # Keep schema in the truncated prompt
        prompt = f"""You are a telecommunications data expert. Analyze the following Encoded TRL (Traffic Record Log) data and provide a comprehensive field analysis.

TEXT CONTENT:
{text}

USER'S QUESTION: {user_question}

TASK: Please analyze this Encoded TRL data and provide:
1. **All Field Names Found**: List every field name present in the data structure
2. **Missing Standard Fields**: Based on the schema below, identify fields that should be present but are missing

Standard TRL fields that should typically be present:
SCHEMA CONTENT:
{schema}

Please provide a detailed analysis of what fields are present vs. missing from the expected schema."""
        print(f"📊 New estimated tokens: {estimate_tokens(prompt):,}")

# Remove unused variables and comments for FAQ generation

messages = [
    {"role": "system", "content": "You are a helpful AI assistant that answers questions based on provided text content. Be accurate and cite information from the text when possible."},
    {"role": "user", "content": prompt}
]

chatgpt_payload = {
    "model": "gpt-3.5-turbo",  # Use standard model instead of 16k
    "messages": messages,
    "temperature": 0.7,  # Lower temperature for more focused answers
    "max_tokens": 500,   # Reduced to be safer
    "top_p": 1
}

import requests
import time
import random

def make_request_with_retry(url, payload, headers, max_retries=3):
    """Make API request with exponential backoff retry logic"""
   
    # Debug: Print payload size
    payload_str = json.dumps(payload)
    print(f"🔍 Debug - Payload size: {len(payload_str)} characters")
   
    for attempt in range(max_retries + 1):
        try:
            response = requests.request("POST", url, json=payload, headers=headers, timeout=30)
           
            if response.status_code == 429:  # Rate limit error
                if attempt < max_retries:
                    # Exponential backoff: 2^attempt + random jitter
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"Rate limit hit. Retrying in {wait_time:.2f} seconds... (Attempt {attempt + 1}/{max_retries + 1})")
                    time.sleep(wait_time)
                    continue
                else:
                    print("Maximum retries reached. Please wait longer before trying again.")
                    return None
           
            response.raise_for_status()  # Raises an HTTPError for other bad responses
            return response.json()
           
        except requests.exceptions.ConnectionError:
            if attempt < max_retries:
                wait_time = (2 ** attempt) + random.uniform(0, 1)
                print(f"Connection failed. Retrying in {wait_time:.2f} seconds... (Attempt {attempt + 1}/{max_retries + 1})")
                time.sleep(wait_time)
                continue
            else:
                print("ERROR: Connection failed after all retries. Check your internet connection.")
                return None
               
        except requests.exceptions.Timeout:
            if attempt < max_retries:
                wait_time = (2 ** attempt) + random.uniform(0, 1)
                print(f"Request timed out. Retrying in {wait_time:.2f} seconds... (Attempt {attempt + 1}/{max_retries + 1})")
                time.sleep(wait_time)
                continue
            else:
                print("ERROR: Request timed out after all retries.")
                return None
               
        except requests.exceptions.HTTPError as e:
            print(f"ERROR: HTTP error occurred: {e}")
            if response.status_code == 401:
                print("Check your API key in the .env file")
            elif response.status_code == 400:
                print("Bad Request - likely the content is too large or contains invalid characters")
                print("Try with a smaller file or check the file format")
            return None
           
        except Exception as e:
            print(f"ERROR: Unexpected error: {e}")
            return None
   
    return None

# Make the API request with retry logic
print(f"\nAnalyzing TRL/CDR file structure...")
print("Identifying missing schema fields...")
print("Generating answer based on the loaded text...")
response_data = make_request_with_retry(chatgpt_url, chatgpt_payload, chatgpt_headers)

if response_data:
    try:
        print("\n" + "="*60)
        print("SCHEMA ANALYSIS RESULTS:")
        print("="*60)
        answer = response_data['choices'][0]['message']['content']
        print(answer)
        print("="*60)
        print("\n✅ Schema analysis completed successfully!")
        # Ask if user wants to ask another question
        print("\nWould you like to ask another question about this text? (y/n)")
        continue_asking = input("Continue? ").strip().lower()
       
        if continue_asking in ['y', 'yes']:
            # Restart the question loop
            while True:
                user_question = get_user_question()
               
                # Use RAG if Pinecone is available
                relevant_context = ""
                if use_pinecone:
                    print("🔍 Searching vector database for relevant context...")
                    contexts = query_pinecone(user_question, top_k=5)
                   
                    if contexts:
                        print(f"✅ Found {len(contexts)} relevant chunks")
                        relevant_context = "\n\nRELEVANT CONTEXT FROM VECTOR DATABASE:\n"
                        for i, ctx in enumerate(contexts, 1):
                            relevant_context += f"\n[Context {i}] (Similarity: {ctx['score']:.3f})\n{ctx['text']}\n"
                    else:
                        print("⚠️  No relevant context found in vector database")
               
                # Update the prompt with new question and RAG context
                prompt = f"""Based on the following text content, please answer the user's question:

TEXT CONTENT:
{text}{relevant_context}

USER'S QUESTION: {user_question}

Please provide a comprehensive and accurate answer based on the information provided above. If the answer cannot be found in the text, please indicate that."""
               
                messages = [
                    {"role": "system", "content": "You are a helpful AI assistant that answers questions based on provided text content. Be accurate and cite information from the text when possible."},
                    {"role": "user", "content": prompt}
                ]
               
                chatgpt_payload["messages"] = messages
               
                print(f"\nProcessing your question: '{user_question}'")
                print("Generating answer based on the loaded text...")
                response_data = make_request_with_retry(chatgpt_url, chatgpt_payload, chatgpt_headers)
               
                if response_data:
                    print("\n" + "="*60)
                    print("ANSWER:")
                    print("="*60)
                    answer = response_data['choices'][0]['message']['content']
                    print(answer)
                    print("="*60)
                   
                    print("\nWould you like to ask another question about this text? (y/n)")
                    continue_asking = input("Continue? ").strip().lower()
                   
                    if continue_asking not in ['y', 'yes']:
                        print("Thank you for using the Q&A system!")
                        break
                else:
                    print("Failed to get response from OpenAI API.")
                    break
        else:
            print("Thank you for using the Q&A system!")
       
    except KeyError as e:
        print(f"ERROR: Unexpected response format: {e}")
        print(f"Response data: {response_data}")
else:
    print("Failed to get response from OpenAI API.")

